# Hadamard edge detection

Encode the full 128 × 128 photo and sample the cyclic edge detector.

Set up local FlagQuantum execution and image plotting.
## The quantum part of the image workflow

The native program prepares all 16,384 image amplitudes, then computes both
cyclic differences through controlled integer decrement and interference.
The high seven bits of the raw unsigned image word encode the row; the low
seven encode the column. Quantum numeric slices count from the least significant
bit. The flags retain both difference branches and their combined edge event.

Users supply normalized image data, logical registers and a resource budget;
synth constructs the loader and shift gates. The separate Circuit loader in
[quantum.py](quantum.py) is retained as an independent preparation reference.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.image_processing.quantum_hadamard_edge_detection.overview import (
    workflow,
)
from examples.synthesis.applications.image_processing.quantum_hadamard_edge_detection.overview.main import (
    print_report,
    save_figures,
)
from examples.synthesis.applications.image_processing.quantum_hadamard_edge_detection.overview.quantum import (
    AXIS_BITS,
    decrement,
)
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QArray,
    QBit,
    QNum,
    X,
    allocate,
    control,
    prepare_amplitudes,
    qfunc,
)


@qfunc
def detect_edges(
    amplitudes: list[float],
    image: Output[QNum[14]],
    flags: Output[QArray[QBit]],
) -> None:
    prepare_amplitudes(amplitudes, 0, image)
    allocate(3, flags)
    H(flags[0])
    control(flags[0], lambda: decrement(image[7:14]))
    H(flags[0])
    H(flags[1])
    control(flags[1], lambda: decrement(image[0:7]))
    H(flags[1])
    control(flags[0], lambda: X(flags[2]))
    control(flags[1], lambda: X(flags[2]))
    control(flags[0:2], lambda: X(flags[2]))


Load the original RGB asset.

In [ ]:
photo = workflow.load_photo()
print('Photo:', tuple(photo.shape))

Reproduce the original posterization preview. Keep the source photo for amplitude encoding.

In [ ]:
preview = workflow.threshold_preview(photo)
assert (preview[0] == photo[0]).all()

Convert unmodified pixel magnitudes to unit amplitudes.

In [ ]:
pixels = workflow.pixel_amplitudes(photo)
print('Amplitude norm:', float(pixels.norm()))

Compute the original seven bits along each axis.

In [ ]:
print('Pixel qubits along each axis:', AXIS_BITS)

Preserve all 16,384 pixel amplitudes.

In [ ]:
print('Pixel count:', pixels.numel())
assert pixels.numel() == 16_384

Prepare the real amplitude vector and compose both cyclic shifts and their edge flag.

In [ ]:
detector_program = synth.synthesize(
    detect_edges, classical_inputs={"amplitudes": pixels.flatten().tolist()},
    max_qubits=20, max_workspace=3,
    max_operations=35000, memory_limit_bytes=64*1024**2,
)
print("Complete native program:", detector_program.report["resources"])
result = workflow.run()
print('Complete state error:', result.state_error)


Check the three restored compiler work bits before taking the direction-bit marginal.

In [ ]:
print('Work probability:', result.work_probability)
assert result.work_probability < 1e-24

Apply the source width bound to the complete circuit.

In [ ]:
assert result.resources.n_qubits <= 20

Report the actual prepared and composed resources.

In [ ]:
print_report(result)

Inspect the emitted rotation and controlled shift resources.

In [ ]:
print('Preparation operations:', dict(result.preparation_resources.operation_counts))
print('Full circuit operations:', dict(result.resources.operation_counts))

Retain all 200,000 finite local draws and report actual edge-flag events.

In [ ]:
print('Total draws:', workflow.SHOTS)
print('Observed edge events:', result.observed_edge_shots)

Reconstruct the sampled image from square roots of measured joint frequencies.

In [ ]:
print('Raw nonzero edge pixels:', int((result.sampled_edge > 0).sum()))

Plot the sampled pixel magnitude histogram.

In [ ]:
figures = Path('notebook-figures')
save_figures(result, figures)
display(Image(filename=str(figures / 'edge-histograms.png')))

Threshold the joint magnitudes at 0.003 and inspect the thresholded histogram.

In [ ]:
print('Edge threshold:', workflow.THRESHOLD)
assert bool(((result.thresholded_edge == 0) | (result.thresholded_edge > workflow.THRESHOLD)).all())

Display the transposed finite-shot result with the exact edge comparison.

In [ ]:
display(Image(filename=str(figures / 'edge-images.png')))

Check that the original photo stayed unmodified through the full workflow.

Try changing only `workflow.THRESHOLD` for sampled-image postprocessing. Changing
the quantum shift requires editing and validating the detector, not just changing
a plot. The complete complex-state stencil check protects coherent interference;
200,000 seeded draws separately demonstrate finite sampling.


In [ ]:
assert bool((result.photo == photo).all())
print('Original photo retained:', tuple(result.photo.shape))